# 课后练习解答（03.05_amp_warmup_tuning）

本解答对应《AMP 混合精度与 Warmup 调优》课后练习，共 15 题。


### 问题1（单选题）

**题目：** AMP 混合精度训练的主要目的是什么？

A. 减少训练代码行数
B. 在保持精度可接受的前提下降低显存占用并提升计算效率
C. 把图片转为 XML
D. 替代优化器

**解答：** B

**解析：** AMP 通过部分算子使用低精度计算来提升性能和降低显存占用。


### 问题2（单选题）

**题目：** Warmup 学习率策略的核心作用是？

A. 训练初期逐步增大学习率，降低不稳定风险
B. 永久冻结模型参数
C. 删除低分目标框
D. 关闭 NPU

**解答：** A

**解析：** Warmup 可以避免训练初期学习率过大导致梯度不稳定。


### 问题3（单选题）

**题目：** 当训练出现 NPU 显存不足时，通常最先尝试调整的是？

A. 增大 batch size
B. 减小 batch size
C. 删除验证集
D. 关闭日志目录

**解答：** B

**解析：** batch size 直接影响激活和梯度占用，是最常见的 OOM 调整项。


### 问题4（单选题）

**题目：** DataLoader workers 主要影响哪一部分？

A. 数据读取与预处理并行度
B. 模型类别名称
C. Git 远端地址
D. ATC 转换精度

**解答：** A

**解析：** workers 数量影响 CPU 侧加载和预处理速度。


### 问题5（多选题）

**题目：** AMP 训练通常会涉及哪些组件或概念？

A. autocast
B. GradScaler
C. float16/bfloat16
D. 反向传播缩放

**解答：** A、B、C、D

**解析：** AMP 不只是换 dtype，还包括自动混合精度上下文和梯度缩放等机制。


### 问题6（多选题）

**题目：** 进行超参数调优时，应优先记录哪些指标？

A. loss 曲线
B. throughput 或 step time
C. 显存/NPU 使用情况
D. checkpoint 是否正常保存

**解答：** A、B、C、D

**解析：** 这些指标共同反映收敛、性能和稳定性。


### 问题7（多选题）

**题目：** 训练不稳定时，可以尝试哪些调整？

A. 降低学习率
B. 增加 warmup epoch
C. 关闭或调整 AMP
D. 检查标注数据是否异常

**解答：** A、B、C、D

**解析：** 训练不稳定可能来自优化参数，也可能来自数据或精度设置。


### 问题8（判断题）

**题目：** AMP 一定会让所有模型训练结果完全不变。

**解答：** 错误

**解析：** AMP 可能带来数值差异，需要通过 loss 和验证指标确认稳定性。


### 问题9（判断题）

**题目：** 调优时一次只改一个关键参数，更容易判断变化来源。

**解答：** 正确

**解析：** 同时修改多个变量会让结果难以归因。


### 问题10（填空题）

**题目：** 训练初期逐步提升学习率的策略通常称为 `____`。

**解答：** Warmup

**解析：** Warmup 是深度学习训练中常见的稳定化策略。


### 问题11（填空题）

**题目：** 增大 `batch_size` 往往会提高吞吐，但也会增加 `____` 占用。

**解答：** 显存或 NPU 内存

**解析：** 更大的 batch 会带来更多中间激活和梯度存储。


### 问题12（简答题）

**题目：** 为什么不能只用 throughput 判断训练是否更好？

**解答：** throughput 只表示速度，不表示模型是否收敛或精度是否正常。实验报告应同时比较 loss、稳定性、显存占用和 checkpoint。

**解析：** 性能优化必须兼顾正确性。


### 问题13（简答题）

**题目：** Warmup 与 Cosine 学习率组合有什么意义？

**解答：** Warmup 负责训练初期平滑升高学习率，Cosine 负责后期逐步衰减学习率，组合起来兼顾启动稳定性和后期收敛。

**解析：** 这是检测模型训练中常见的学习率调度思路。


### 问题14（简答题）

**题目：** 如果开启 AMP 后 loss 变成 NaN，应如何排查？

**解答：** 先降低学习率或增大 warmup，再确认 GradScaler 是否正确使用，检查输入数据和标注是否存在异常，必要时暂时关闭 AMP 对比。

**解析：** NaN 可能由数值溢出、异常数据或过激学习率引起。


### 问题15（代码设计题）

**题目：** 写出 AMP 训练中 autocast 和 GradScaler 的核心用法。

**解答：** ```python
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
for images, targets in loader:
    optimizer.zero_grad()
    with torch.autocast(device_type="npu", enabled=use_amp):
        outputs = model(images)
        loss = yolo_loss(outputs, targets)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
```

**解析：** 实际 Ascend 环境中 autocast/GradScaler 的接口可能随版本略有差异，应以当前 torch_npu 支持为准。
